In [1]:
import os
import sys

# ==================================================
# PROJECT
# ==================================================

PROJECT_ROOT = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing"
)

os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(
        0,
        PROJECT_ROOT
    )


# ==================================================
# IMPORT V2
# ==================================================

from app.router.adaptive import (
    AdaptiveRouterV2
)


# ==================================================
# CREATE V2 ROUTER
# ==================================================

router = AdaptiveRouterV2(
    label_cache_path=(
        "notebooks/data/evaluation/"
        "v1_complexity_labels.json"
    )
)


# ==================================================
# LOAD DOLLY
# ==================================================

dataset = router.load_data()


# ==================================================
# REUSE V1 LABELS
# ==================================================

queries, labels = (
    router.prepare_v2_data(
        dataset
    )
)


# ==================================================
# TRAIN V2
# ==================================================

results = router.fit(
    queries,
    labels
)


# ==================================================
# V1 VS V2
# ==================================================

v1_accuracy = 0.6016

v2_accuracy = (
    results["accuracy"]
)

difference = (
    v2_accuracy
    - v1_accuracy
)

print("\n" + "=" * 60)
print("V1 VS V2")
print("=" * 60)

print(
    f"V1 Accuracy : "
    f"{v1_accuracy:.4f}"
)

print(
    f"V2 Accuracy : "
    f"{v2_accuracy:.4f}"
)

print(
    f"Difference  : "
    f"{difference:+.4f}"
)

print(
    f"Percentage-point change: "
    f"{difference * 100:+.2f}"
)

/home/sagemaker-user/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Dolly dataset...


Dataset size: 15011
Columns: ['instruction', 'context', 'response', 'category']

Using existing V1 complexity labels.
Label file: notebooks/data/evaluation/v1_complexity_labels.json

Existing V1 labels found.
Loading: notebooks/data/evaluation/v1_complexity_labels.json
Loaded 639 labeled examples.

Prepared examples: 639

Complexity distribution:
  simple  : 213
  medium  : 213
  complex : 213

V2 examples: 639

TRAINING ADAPTIVE ROUTER V2

Training samples: 511
Validation samples: 128

Generating training embeddings...

LOADING V2 EMBEDDING MODEL

Model: sentence-transformers/all-MiniLM-L6-v2
Loading sentence-transformer...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8820.02it/s]


Embedding model loaded.


Batches: 100%|██████████| 16/16 [00:03<00:00,  4.45it/s]



Training embedding shape:
(511, 384)

Generating validation embeddings...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.39it/s]


Validation embedding shape:
(128, 384)

Training Logistic Regression...

V2 VALIDATION RESULTS

Accuracy: 0.4922

Classification Report:
              precision    recall  f1-score   support

      simple       0.48      0.47      0.47        43
      medium       0.46      0.60      0.52        43
     complex       0.59      0.40      0.48        42

    accuracy                           0.49       128
   macro avg       0.51      0.49      0.49       128
weighted avg       0.51      0.49      0.49       128


Confusion Matrix:
[[20 18  5]
 [10 26  7]
 [12 13 17]]

V1 VS V2
V1 Accuracy : 0.6016
V2 Accuracy : 0.4922
Difference  : -0.1094
Percentage-point change: -10.94


In [3]:
import os
import sys

PROJECT_ROOT = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing"
)

os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(
        0,
        PROJECT_ROOT
    )

from app.evaluation.audit_labels import (
    LabelAuditor
)

# --------------------------------------------------
# Create auditor
# --------------------------------------------------

auditor = LabelAuditor(
    label_path=(
        "notebooks/data/evaluation/"
        "v1_complexity_labels.json"
    )
)

# --------------------------------------------------
# Load
# --------------------------------------------------

labels = auditor.load_labels()

# --------------------------------------------------
# Audit
# --------------------------------------------------

df = auditor.audit(
    labels
)

# --------------------------------------------------
# Create review CSV
# --------------------------------------------------

review_file = (
    auditor.create_review_file(
        df
    )
)

# --------------------------------------------------
# Create low-confidence subset
# --------------------------------------------------

low_confidence = (
    auditor.create_low_confidence_file(
        df,
        threshold=0.65
    )
)

Loading labels:
notebooks/data/evaluation/v1_complexity_labels.json
Loaded 639 examples.

LABEL AUDIT

Total examples: 639

Complexity distribution:
complexity
complex    213
medium     213
simple     213
Name: count, dtype: int64

Confidence statistics:
count    639.000000
mean       0.447165
std        0.073500
min        0.335859
25%        0.397412
50%        0.435278
75%        0.476351
max        0.969588
Name: confidence, dtype: float64

Missing values:
instruction    0
complexity     0
confidence     0
dtype: int64

Review file created:
notebooks/data/evaluation/label_audit/label_review.csv

Low-confidence examples: 624
Saved to:
notebooks/data/evaluation/label_audit/low_confidence_labels.csv


In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# V1 LABEL AUDIT — SINGLE NOTEBOOK CELL
# ============================================================

PROJECT_ROOT = Path("/home/sagemaker-user/adaptive-llm-routing")

REVIEW_FILE = (
    PROJECT_ROOT
    / "notebooks/data/evaluation/label_audit/label_review.csv"
)

# Load audit file
if not REVIEW_FILE.exists():
    raise FileNotFoundError(
        f"Audit file not found:\n{REVIEW_FILE}\n\n"
        "Run app/evaluation/audit_labels.py first."
    )

review = pd.read_csv(REVIEW_FILE)

# Display settings
pd.set_option("display.max_colwidth", 500)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("=" * 100)
print("V1 LABEL AUDIT")
print("=" * 100)

print(f"\nTotal examples: {len(review)}")

print("\nCurrent label distribution:")
print(review["complexity"].value_counts().sort_index())

print("\nConfidence statistics:")
print(review["confidence"].describe())

# ------------------------------------------------------------
# LOWEST CONFIDENCE — MOST IMPORTANT TO REVIEW
# ------------------------------------------------------------

lowest = (
    review
    .sort_values("confidence", ascending=True)
    .head(50)
    .copy()
)

print("\n" + "=" * 100)
print("50 LOWEST-CONFIDENCE EXAMPLES")
print("=" * 100)

display(
    lowest[
        [
            "instruction",
            "complexity",
            "confidence"
        ]
    ].reset_index(drop=True)
)

# ------------------------------------------------------------
# HIGHEST CONFIDENCE — CONTROL GROUP
# ------------------------------------------------------------

highest = (
    review
    .sort_values("confidence", ascending=False)
    .head(50)
    .copy()
)

print("\n" + "=" * 100)
print("50 HIGHEST-CONFIDENCE EXAMPLES")
print("=" * 100)

display(
    highest[
        [
            "instruction",
            "complexity",
            "confidence"
        ]
    ].reset_index(drop=True)
)

# ------------------------------------------------------------
# CONFIDENCE BUCKETS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CONFIDENCE BUCKETS")
print("=" * 100)

review["confidence_bucket"] = pd.cut(
    review["confidence"],
    bins=[0, 0.40, 0.50, 0.65, 0.80, 1.0],
    labels=[
        "<0.40",
        "0.40–0.50",
        "0.50–0.65",
        "0.65–0.80",
        "0.80–1.00"
    ],
    include_lowest=True
)

print(
    review["confidence_bucket"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 100)
print("IMPORTANT")
print("=" * 100)
print(
    "Do NOT automatically change labels based only on confidence.\n"
    "The original V1 label file remains untouched.\n"
    "Use the examples above to establish a consistent complexity rubric "
    "before creating corrected labels."
)

In [2]:
test_queries = [
    "What is Python?",
    "What is the capital of France?",
    "Explain how neural networks work.",
    "Write a Python function to reverse a linked list.",
    "What is supervised learning?",
    "Design a fraud detection system using graph neural networks.",
    "Explain the difference between TCP and UDP.",
    "Build a REST API using FastAPI and PostgreSQL.",
    "How does backpropagation work?",
    "Create an end-to-end machine learning pipeline for fraud detection."
]

print("=" * 70)
print("V2 ROUTING TEST")
print("=" * 70)

for query in test_queries:

    result = router.route_model(query)

    print("\nQuery:", query)
    print("Complexity:", result["complexity"])
    print("Confidence:", round(result["confidence"], 4))
    print("Model:", result["model"])
    print("Cached:", result["cached"])

V2 ROUTING TEST

Query: What is Python?
Complexity: medium
Confidence: 0.402
Model: medium_model
Cached: False

Query: What is the capital of France?
Complexity: complex
Confidence: 0.5046
Model: large_model
Cached: False

Query: Explain how neural networks work.
Complexity: simple
Confidence: 0.7166
Model: small_model
Cached: False

Query: Write a Python function to reverse a linked list.
Complexity: simple
Confidence: 0.5095
Model: small_model
Cached: False

Query: What is supervised learning?
Complexity: simple
Confidence: 0.4576
Model: small_model
Cached: False

Query: Design a fraud detection system using graph neural networks.
Complexity: simple
Confidence: 0.5273
Model: small_model
Cached: False

Query: Explain the difference between TCP and UDP.
Complexity: simple
Confidence: 0.3991
Model: small_model
Cached: False

Query: Build a REST API using FastAPI and PostgreSQL.
Complexity: medium
Confidence: 0.4393
Model: medium_model
Cached: False



Query: How does backpropagation work?
Complexity: simple
Confidence: 0.6817
Model: small_model
Cached: False

Query: Create an end-to-end machine learning pipeline for fraud detection.
Complexity: medium
Confidence: 0.4605
Model: medium_model
Cached: False


In [4]:
import pandas as pd
from pathlib import Path

# ============================================================
# V1 LABEL AUDIT — SINGLE NOTEBOOK CELL
# ============================================================

PROJECT_ROOT = Path("/home/sagemaker-user/adaptive-llm-routing")

REVIEW_FILE = (
    PROJECT_ROOT
    / "notebooks/data/evaluation/label_audit/label_review.csv"
)

# Load audit file
if not REVIEW_FILE.exists():
    raise FileNotFoundError(
        f"Audit file not found:\n{REVIEW_FILE}\n\n"
        "Run app/evaluation/audit_labels.py first."
    )

review = pd.read_csv(REVIEW_FILE)

# Display settings
pd.set_option("display.max_colwidth", 500)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("=" * 100)
print("V1 LABEL AUDIT")
print("=" * 100)

print(f"\nTotal examples: {len(review)}")

print("\nCurrent label distribution:")
print(review["complexity"].value_counts().sort_index())

print("\nConfidence statistics:")
print(review["confidence"].describe())

# ------------------------------------------------------------
# LOWEST CONFIDENCE — MOST IMPORTANT TO REVIEW
# ------------------------------------------------------------

lowest = (
    review
    .sort_values("confidence", ascending=True)
    .head(50)
    .copy()
)

print("\n" + "=" * 100)
print("50 LOWEST-CONFIDENCE EXAMPLES")
print("=" * 100)

display(
    lowest[
        [
            "instruction",
            "complexity",
            "confidence"
        ]
    ].reset_index(drop=True)
)

# ------------------------------------------------------------
# HIGHEST CONFIDENCE — CONTROL GROUP
# ------------------------------------------------------------

highest = (
    review
    .sort_values("confidence", ascending=False)
    .head(50)
    .copy()
)

print("\n" + "=" * 100)
print("50 HIGHEST-CONFIDENCE EXAMPLES")
print("=" * 100)

display(
    highest[
        [
            "instruction",
            "complexity",
            "confidence"
        ]
    ].reset_index(drop=True)
)

# ------------------------------------------------------------
# CONFIDENCE BUCKETS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CONFIDENCE BUCKETS")
print("=" * 100)

review["confidence_bucket"] = pd.cut(
    review["confidence"],
    bins=[0, 0.40, 0.50, 0.65, 0.80, 1.0],
    labels=[
        "<0.40",
        "0.40–0.50",
        "0.50–0.65",
        "0.65–0.80",
        "0.80–1.00"
    ],
    include_lowest=True
)

print(
    review["confidence_bucket"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 100)
print("IMPORTANT")
print("=" * 100)
print(
    "Do NOT automatically change labels based only on confidence.\n"
    "The original V1 label file remains untouched.\n"
    "Use the examples above to establish a consistent complexity rubric "
    "before creating corrected labels."
)

V1 LABEL AUDIT

Total examples: 639

Current label distribution:
complexity
complex    213
medium     213
simple     213
Name: count, dtype: int64

Confidence statistics:
count    639.000000
mean       0.447165
std        0.073500
min        0.335859
25%        0.397412
50%        0.435278
75%        0.476351
max        0.969588
Name: confidence, dtype: float64

50 LOWEST-CONFIDENCE EXAMPLES


,instruction,complexity,confidence
0,"Given a reference text about Gennady Petrovich Lyachin, tell me when he joined the navy and when he was decommissioned.",simple,0.335859
1,Extract from the text the main characteristics of the city of São Paulo. Separate them with a comma.,simple,0.337699
2,List the years in which Anne Rochelle Steely Ramirez participated in Sprint events based on the passage. Use bulleted format to represent the years and sort in ascending order.,simple,0.339872
3,"Identify all PII in this customer review ""My name is Joe Smith and email address is joesmith9999@gmail.com. I'd like to leave a review about Langton's red wine. It was tasty for a red, but there were sediments at the bottom of my bottle""",simple,0.340561
4,What is a newspaper?,simple,0.342671
5,Are UGGs considered fashionable?,simple,0.343441
6,Provide a list of diseases which spread in the Diamond Princess and the virus that caused them. Use bulleted list in the format ‘Disease Name - Virus Name - year’.,simple,0.345798
7,What is Python?,simple,0.346295
8,"From the passage provided, extract the programming languages supported by Flink. Separate them with a comma.",medium,0.346366
9,Name some famous rock bands from the 1960s,medium,0.347502



50 HIGHEST-CONFIDENCE EXAMPLES


,instruction,complexity,confidence
0,Write a short story about a person who discovers a hidden room in their house. The story should include a plot twist and a clear resolution at the end.,simple,0.969588
1,Give me 3 ideas for hobbies that I can recommend to somebody who is retiring. These should all be activities that require gentle movement and are not overly demanding on the body.,medium,0.824767
2,"Hello, my family and I recently went to the zoo and saw a number of animals. Please identify the animals in this summary and organize them in order of size. My son's favorite animal is an elephant, and he saw an elephant in the grassland. My wife caught a parakeet trying to eat our popcorn. My daughter was scared by lion that she saw on the prairie. I missed all of these animals and only saw a boa constrictor in an a cage.",medium,0.784785
3,"From the passage provided, extract the languages in which Kishore Kumar provided his vocals as a playback singer. Separate them with a comma.",simple,0.746946
4,Please provide a short biography of The Needle from the passage provided.,simple,0.722473
5,"I dont think I should pursue higher education. From the following, what are some pros and cons of getting a masters education?\n\ndegree to show mastery of a subject\nTakes money\nHard work is required\nLosing out on entrepreneurial opportunities by the time spent studying\ndegree to show mastery of a subject\nskill development\nTime away from family\nSet you up for lifelong success\nrespect in society",medium,0.718135
6,What types of problems in government could AI be used to solve,complex,0.684676
7,"First you'd need to consider what you're using it for. Exercise bikes can be very different from commuters or casual ride bikes or velodrome racers. Another consideration is budget: bikes can range from a couple hundred dollars in price to several thousand. Accessories can dramatically increase the price as well.\n\nFor general, all-purpose bikes you may want to consider a commuter bicycle. It would have a wider, thicker wheel than a road bike so it would be a little sturdier for urban roads...",simple,0.677175
8,"Categorize the following list into common greetings and common parting words: goodbye, hi, hiya, howdy, see ya, what's up, i'm out, how are you, how are you doing, peace, sup, talk to you later, see you soon, have a good one",simple,0.673058
9,"Provide a short review for a Japanese restaurant called Honto in Brisbane. It was a little hard to find as it is in a backstreet, but we found it without too much trouble. Be sure to mention it was very atmospheric and that the food was great.",medium,0.671330



CONFIDENCE BUCKETS
confidence_bucket
<0.40        168
0.40–0.50    357
0.50–0.65     99
0.65–0.80     13
0.80–1.00      2
Name: count, dtype: int64

IMPORTANT
Do NOT automatically change labels based only on confidence.
The original V1 label file remains untouched.
Use the examples above to establish a consistent complexity rubric before creating corrected labels.


In [5]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/adaptive-llm-routing")

INPUT_FILE = (
    PROJECT_ROOT /
    "notebooks/data/evaluation/label_audit/label_review.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "notebooks/data/evaluation/label_audit"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

review = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. Lowest-confidence examples
# ------------------------------------------------------------

low_conf = (
    review
    .sort_values("confidence", ascending=True)
    .head(100)
    .copy()
)

# ------------------------------------------------------------
# 2. Random control sample from high-confidence examples
# ------------------------------------------------------------

high_conf_pool = review[
    ~review["instruction"].isin(low_conf["instruction"])
].copy()

high_conf = (
    high_conf_pool
    .sort_values("confidence", ascending=False)
    .head(25)
    .copy()
)

# ------------------------------------------------------------
# 3. Combine
# ------------------------------------------------------------

audit = pd.concat(
    [low_conf, high_conf],
    ignore_index=True
)

# Add columns for human review
audit["review_label"] = ""
audit["review_status"] = "pending"
audit["review_notes"] = ""

# Give each row a stable review ID
audit.insert(
    0,
    "review_id",
    range(1, len(audit) + 1)
)

# Keep only useful columns
audit = audit[
    [
        "review_id",
        "instruction",
        "complexity",
        "confidence",
        "review_label",
        "review_status",
        "review_notes"
    ]
]

OUTPUT_FILE = OUTPUT_DIR / "v1_label_review_125.csv"

audit.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("LABEL REVIEW FILE CREATED")
print("=" * 70)

print(f"Total examples: {len(audit)}")
print(f"Low-confidence examples: {len(low_conf)}")
print(f"High-confidence control examples: {len(high_conf)}")

print(f"\nSaved to:")
print(OUTPUT_FILE)

print("\nReview instructions:")
print("review_label must be: simple / medium / complex")
print("review_status should be: corrected / confirmed")
print("review_notes is optional.")

display(audit.head(20))

LABEL REVIEW FILE CREATED
Total examples: 125
Low-confidence examples: 100
High-confidence control examples: 25

Saved to:
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/label_audit/v1_label_review_125.csv

Review instructions:
review_label must be: simple / medium / complex
review_status should be: corrected / confirmed
review_notes is optional.


,review_id,instruction,complexity,confidence,review_label,review_status,review_notes
0,1,"Given a reference text about Gennady Petrovich Lyachin, tell me when he joined the navy and when he was decommissioned.",simple,0.335859,,pending,
1,2,Extract from the text the main characteristics of the city of São Paulo. Separate them with a comma.,simple,0.337699,,pending,
2,3,List the years in which Anne Rochelle Steely Ramirez participated in Sprint events based on the passage. Use bulleted format to represent the years and sort in ascending order.,simple,0.339872,,pending,
3,4,"Identify all PII in this customer review ""My name is Joe Smith and email address is joesmith9999@gmail.com. I'd like to leave a review about Langton's red wine. It was tasty for a red, but there were sediments at the bottom of my bottle""",simple,0.340561,,pending,
4,5,What is a newspaper?,simple,0.342671,,pending,
5,6,Are UGGs considered fashionable?,simple,0.343441,,pending,
6,7,Provide a list of diseases which spread in the Diamond Princess and the virus that caused them. Use bulleted list in the format ‘Disease Name - Virus Name - year’.,simple,0.345798,,pending,
7,8,What is Python?,simple,0.346295,,pending,
8,9,"From the passage provided, extract the programming languages supported by Flink. Separate them with a comma.",medium,0.346366,,pending,
9,10,Name some famous rock bands from the 1960s,medium,0.347502,,pending,


In [8]:
# ============================================================
# V1.1 LABEL REVIEW — MODEL-ASSISTED SECOND PASS
# ============================================================
#
# Purpose:
#   1. Load the 125 examples selected for review
#   2. Use a stronger zero-shot model to propose complexity
#   3. Compare original V1 label vs proposed label
#   4. Save a review CSV
#
# IMPORTANT:
#   - Original V1 labels are NEVER modified
#   - Proposed labels are NOT treated as ground truth
#   - No keyword-based complexity rules are used
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from transformers import pipeline
import torch

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/home/sagemaker-user/adaptive-llm-routing")

INPUT_FILE = (
    PROJECT_ROOT /
    "notebooks/data/evaluation/label_audit/v1_label_review_125.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "notebooks/data/evaluation/label_audit"
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "v1_label_review_125_proposed.csv"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# LOAD REVIEW DATA
# ------------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Review file not found:\n{INPUT_FILE}"
    )

review = pd.read_csv(INPUT_FILE)

print("=" * 80)
print("LOADED REVIEW DATA")
print("=" * 80)

print(f"Examples: {len(review)}")

# ------------------------------------------------------------
# LOAD ZERO-SHOT MODEL
# ------------------------------------------------------------

MODEL_NAME = "MoritzLaurer/deberta-v3-large-zeroshot-v2.0"

device = 0 if torch.cuda.is_available() else -1

print("\n" + "=" * 80)
print("LOADING ZERO-SHOT MODEL")
print("=" * 80)

print(f"Model : {MODEL_NAME}")
print(
    f"Device: {'CUDA' if device == 0 else 'CPU'}"
)

classifier = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME,
    device=device
)

print("Model loaded.")

# ------------------------------------------------------------
# COMPLEXITY DEFINITIONS
# ------------------------------------------------------------
#
# These are natural-language descriptions rather than
# technical keyword rules.
# ------------------------------------------------------------

candidate_labels = [
    "simple task: a direct, straightforward request that can usually be answered with one basic step, a definition, extraction, lookup, simple classification, simple calculation, or short response",
    
    "medium task: a request requiring several related pieces of information, comparison, moderate explanation, synthesis, or multiple connected steps",
    
    "complex task: a request requiring substantial multi-step reasoning, difficult analysis, multiple dependent constraints, extensive synthesis, or significant problem solving"
]

label_mapping = {
    candidate_labels[0]: "simple",
    candidate_labels[1]: "medium",
    candidate_labels[2]: "complex"
}

# ------------------------------------------------------------
# RUN SECOND-PASS CLASSIFICATION
# ------------------------------------------------------------

proposed_labels = []
proposed_confidence = []
all_scores = []

print("\n" + "=" * 80)
print("RUNNING SECOND-PASS LABELING")
print("=" * 80)

for i, instruction in enumerate(
    review["instruction"].astype(str)
):

    result = classifier(
        instruction,
        candidate_labels,
        hypothesis_template="This is a {}.",
        multi_label=False
    )

    top_label = result["labels"][0]
    top_score = float(result["scores"][0])

    proposed = label_mapping[top_label]

    # Store all scores for analysis
    score_dict = {
        label_mapping[label]: float(score)
        for label, score
        in zip(result["labels"], result["scores"])
    }

    proposed_labels.append(proposed)
    proposed_confidence.append(top_score)
    all_scores.append(score_dict)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(review)}")

# ------------------------------------------------------------
# ADD RESULTS
# ------------------------------------------------------------

review["proposed_label"] = proposed_labels
review["proposed_confidence"] = proposed_confidence

review["proposed_simple_score"] = [
    x["simple"] for x in all_scores
]

review["proposed_medium_score"] = [
    x["medium"] for x in all_scores
]

review["proposed_complex_score"] = [
    x["complex"] for x in all_scores
]

# ------------------------------------------------------------
# DISAGREEMENT
# ------------------------------------------------------------

review["label_changed"] = (
    review["complexity"]
    != review["proposed_label"]
)

review["agreement"] = np.where(
    review["label_changed"],
    "DISAGREEMENT",
    "AGREEMENT"
)

# ------------------------------------------------------------
# REVIEW STATUS
# ------------------------------------------------------------
#
# DO NOT mark model proposals as confirmed.
# They require inspection.
# ------------------------------------------------------------

review["review_status"] = np.where(
    review["label_changed"],
    "needs_review",
    "model_agrees"
)

review["review_label"] = review["proposed_label"]

review["review_notes"] = np.where(
    review["label_changed"],
    "Second-pass model disagrees with original V1 label.",
    "Second-pass model agrees with original V1 label."
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

review.to_csv(
    OUTPUT_FILE,
    index=False
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SECOND-PASS RESULTS")
print("=" * 80)

print(
    f"\nTotal examples: {len(review)}"
)

print(
    f"Agreement:       {(~review['label_changed']).sum()}"
)

print(
    f"Disagreement:    {review['label_changed'].sum()}"
)

print(
    f"Agreement rate:  "
    f"{(~review['label_changed']).mean():.2%}"
)

print("\nOriginal labels:")
print(
    review["complexity"]
    .value_counts()
    .sort_index()
)

print("\nProposed labels:")
print(
    review["proposed_label"]
    .value_counts()
    .sort_index()
)

print("\n" + "=" * 80)
print("DISAGREEMENTS — REVIEW THESE FIRST")
print("=" * 80)

disagreements = (
    review[
        review["label_changed"]
    ]
    .sort_values(
        "proposed_confidence",
        ascending=False
    )
)

display(
    disagreements[
        [
            "review_id",
            "instruction",
            "complexity",
            "confidence",
            "proposed_label",
            "proposed_confidence",
            "agreement"
        ]
    ]
)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(OUTPUT_FILE)

print(
    "\nIMPORTANT:"
    "\nThe proposed labels are model suggestions."
    "\nReview the disagreements before creating V1.1."
)

LOADED REVIEW DATA
Examples: 125

LOADING ZERO-SHOT MODEL
Model : MoritzLaurer/deberta-v3-large-zeroshot-v2.0
Device: CPU


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 10464.84it/s]


Model loaded.

RUNNING SECOND-PASS LABELING
Processed 10/125
Processed 20/125
Processed 30/125
Processed 40/125
Processed 50/125
Processed 60/125
Processed 70/125
Processed 80/125
Processed 90/125
Processed 100/125
Processed 110/125
Processed 120/125

SECOND-PASS RESULTS

Total examples: 125
Agreement:       40
Disagreement:    85
Agreement rate:  32.00%

Original labels:
complexity
complex    18
medium     41
simple     66
Name: count, dtype: int64

Proposed labels:
proposed_label
medium    94
simple    31
Name: count, dtype: int64

DISAGREEMENTS — REVIEW THESE FIRST


,review_id,instruction,complexity,confidence,proposed_label,proposed_confidence,agreement
102,103,"Hello, my family and I recently went to the zoo and saw a number of animals. Please identify the animals in this summary and organize them in order of size. My son's favorite animal is an elephant, and he saw an elephant in the grassland. My wife caught a parakeet trying to eat our popcorn. My daughter was scared by lion that she saw on the prairie. I missed all of these animals and only saw a boa constrictor in an a cage.",medium,0.784785,simple,0.889269,DISAGREEMENT
109,110,"Provide a short review for a Japanese restaurant called Honto in Brisbane. It was a little hard to find as it is in a backstreet, but we found it without too much trouble. Be sure to mention it was very atmospheric and that the food was great.",medium,0.671330,simple,0.887254,DISAGREEMENT
68,69,From the passage identify the spin-off games release in the universe of Clash of Clans. Display the results in a comma separated format.,simple,0.371441,medium,0.866068,DISAGREEMENT
106,107,What types of problems in government could AI be used to solve,complex,0.684676,medium,0.749598,DISAGREEMENT
103,104,"From the passage provided, extract the languages in which Kishore Kumar provided his vocals as a playback singer. Separate them with a comma.",simple,0.746946,medium,0.747858,DISAGREEMENT
71,72,Why zurich is so beautiful,simple,0.371723,medium,0.739607,DISAGREEMENT
121,122,What are some ways to improve the value of your home?,complex,0.606941,medium,0.736255,DISAGREEMENT
115,116,What are some unique curtain tie backs that you can make yourself?,complex,0.634933,medium,0.733945,DISAGREEMENT
33,34,What causes earthquakes?,simple,0.358585,medium,0.725120,DISAGREEMENT
17,18,"Tell me whether these people are DJs or singers: David Guetta, Avicii, Rihanna, Beyonce, Martin Garrix, Calvin Harris, Taylor Swift",complex,0.351555,medium,0.716727,DISAGREEMENT



SAVED
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/label_audit/v1_label_review_125_proposed.csv

IMPORTANT:
The proposed labels are model suggestions.
Review the disagreements before creating V1.1.


In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

INPUT_PATH = Path(
    "/home/sagemaker-user/adaptive-llm-routing/"
    "notebooks/data/evaluation/label_audit/"
    "v1_label_review_125_proposed.csv"
)

OUTPUT_PATH = Path(
    "/home/sagemaker-user/adaptive-llm-routing/"
    "notebooks/data/evaluation/label_audit/"
    "v1_label_review_125_completed.csv"
)

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(INPUT_PATH)

# Create review columns if they don't exist
if "review_label" not in df.columns:
    df["review_label"] = ""

if "review_status" not in df.columns:
    df["review_status"] = ""

if "review_notes" not in df.columns:
    df["review_notes"] = ""

# ============================================================
# LABELING GUIDE
# ============================================================

print("=" * 80)
print("HUMAN LABEL REVIEW")
print("=" * 80)

print("""
LABELING RULES

SIMPLE:
- Direct definition
- Direct factual lookup
- Extraction from provided text
- Basic classification
- Basic arithmetic
- Straightforward one-step answer

MEDIUM:
- Comparison
- Moderate explanation
- Several related pieces of information
- Some synthesis or interpretation
- Multiple connected steps

COMPLEX:
- Substantial multi-step reasoning
- Multiple dependent constraints
- Difficult analysis
- Extensive synthesis
- Significant problem solving

IMPORTANT:
Do NOT judge complexity based on how technical, obscure, or difficult
the TOPIC sounds.

Examples:
"What is Python?"                         -> SIMPLE
"What causes earthquakes?"                -> SIMPLE
"Extract names from this passage"         -> SIMPLE
"Compare X and Y and explain why"         -> MEDIUM
"Analyze several constraints and choose"  -> COMPLEX

INPUT:
s = simple
m = medium
c = complex
q = quit/save progress
""")

# ============================================================
# RESUME SUPPORT
# ============================================================

valid_labels = {
    "s": "simple",
    "m": "medium",
    "c": "complex"
}

start_index = 0

# Resume from first unfinished row
for i, row in df.iterrows():
    if str(row["review_label"]).strip().lower() not in [
        "simple", "medium", "complex"
    ]:
        start_index = i
        break
else:
    start_index = len(df)

print(f"\nStarting at example {start_index + 1}/{len(df)}")

# ============================================================
# INTERACTIVE REVIEW
# ============================================================

for i in range(start_index, len(df)):

    row = df.iloc[i]

    print("\n" + "=" * 80)
    print(f"EXAMPLE {i + 1}/{len(df)}")
    print("=" * 80)

    print("\nINSTRUCTION:")
    print(row["instruction"])

    print("\n" + "-" * 80)

    print(f"ORIGINAL V1 LABEL:       {row['complexity']}")
    print(f"ORIGINAL CONFIDENCE:     {row['confidence']:.4f}")

    print(f"\nDEBERTA PROPOSED LABEL:  {row['proposed_label']}")
    print(f"DEBERTA CONFIDENCE:      {row['proposed_confidence']:.4f}")

    print(f"\nAGREEMENT:               {row['agreement']}")

    # Ask user
    while True:

        choice = input(
            "\nYour label [s=Simple, m=Medium, c=Complex, "
            "q=Save & Quit]: "
        ).strip().lower()

        if choice == "q":
            df.to_csv(OUTPUT_PATH, index=False)

            print("\n" + "=" * 80)
            print("PROGRESS SAVED")
            print("=" * 80)
            print(f"Saved to:\n{OUTPUT_PATH}")

            reviewed = (
                df["review_label"]
                .astype(str)
                .str.lower()
                .isin(["simple", "medium", "complex"])
                .sum()
            )

            print(f"\nReviewed: {reviewed}/{len(df)}")
            print("Run this same cell again to resume.")

            raise SystemExit

        if choice in valid_labels:
            final_label = valid_labels[choice]
            break

        print("Invalid input. Enter s, m, c, or q.")

    # Store human decision
    df.at[df.index[i], "review_label"] = final_label
    df.at[df.index[i], "review_status"] = "confirmed"

    # Optional note for disagreements
    if final_label != str(row["complexity"]).lower():
        df.at[
            df.index[i],
            "review_notes"
        ] = (
            f"Human review changed original label "
            f"{row['complexity']} -> {final_label}"
        )
    else:
        df.at[
            df.index[i],
            "review_notes"
        ] = "Human review confirmed original label."

    # Save after EVERY example
    df.to_csv(OUTPUT_PATH, index=False)

    print(f"\n✓ Saved. Final label: {final_label}")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("REVIEW COMPLETE")
print("=" * 80)

df.to_csv(OUTPUT_PATH, index=False)

print(f"\nSaved to:")
print(OUTPUT_PATH)

print("\nFinal human labels:")
print(df["review_label"].value_counts())

print("\nChanges from original V1:")
changes = (
    df["review_label"].astype(str).str.lower()
    != df["complexity"].astype(str).str.lower()
)

print(f"Changed:   {changes.sum()}")
print(f"Unchanged: {(~changes).sum()}")

print("\nNext step:")
print("Use the completed CSV to generate v1_corrected_complexity_labels.json.")

HUMAN LABEL REVIEW

LABELING RULES

SIMPLE:
- Direct definition
- Direct factual lookup
- Extraction from provided text
- Basic classification
- Basic arithmetic
- Straightforward one-step answer

MEDIUM:
- Comparison
- Moderate explanation
- Several related pieces of information
- Some synthesis or interpretation
- Multiple connected steps

COMPLEX:
- Substantial multi-step reasoning
- Multiple dependent constraints
- Difficult analysis
- Extensive synthesis
- Significant problem solving

IMPORTANT:
Do NOT judge complexity based on how technical, obscure, or difficult
the TOPIC sounds.

Examples:
"What is Python?"                         -> SIMPLE
"What causes earthquakes?"                -> SIMPLE
"Extract names from this passage"         -> SIMPLE
"Compare X and Y and explain why"         -> MEDIUM
"Analyze several constraints and choose"  -> COMPLEX

INPUT:
s = simple
m = medium
c = complex
q = quit/save progress


Starting at example 126/125

REVIEW COMPLETE

Saved to:
/home/sagem

In [3]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter

# ============================================================
# PATHS
# ============================================================

BASE = Path("/home/sagemaker-user/adaptive-llm-routing")

ORIGINAL_JSON = BASE / "notebooks/data/evaluation/v1_complexity_labels.json"

REVIEW_CSV = (
    BASE / "notebooks/data/evaluation/label_audit/"
    "v1_label_review_125_completed.csv"
)

OUTPUT_JSON = (
    BASE / "notebooks/data/evaluation/"
    "v1_corrected_complexity_labels.json"
)

# ============================================================
# LOAD
# ============================================================

with open(ORIGINAL_JSON, "r", encoding="utf-8") as f:
    original = json.load(f)

review = pd.read_csv(REVIEW_CSV)

print("=" * 80)
print("CHECKING HUMAN REVIEW")
print("=" * 80)

print(f"Original examples: {len(original)}")
print(f"Review rows:       {len(review)}")

# ============================================================
# CHECK REVIEW LABELS
# ============================================================

review["review_label"] = (
    review["review_label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

valid_labels = {"simple", "medium", "complex"}

invalid = review[
    ~review["review_label"].isin(valid_labels)
]

if len(invalid) > 0:
    print("\nInvalid/unreviewed rows:")
    print(
        invalid[
            ["review_id", "review_label"]
        ].to_string(index=False)
    )
    raise ValueError(
        f"{len(invalid)} rows do not have a valid review_label."
    )

print("\nAll review labels are valid.")

# ============================================================
# CREATE CORRECTED COPY
# ============================================================

corrected = original.copy()

changes = 0

for _, row in review.iterrows():

    # review_id is 1-based
    idx = int(row["review_id"]) - 1

    if idx < 0 or idx >= len(corrected):
        raise IndexError(
            f"Invalid review_id: {row['review_id']}"
        )

    old_label = str(corrected[idx]["complexity"]).lower()
    new_label = row["review_label"]

    if old_label != new_label:
        changes += 1

    corrected[idx]["complexity"] = new_label
    corrected[idx]["label_source"] = "human_review"

# ============================================================
# SAVE
# ============================================================

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        corrected,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# SUMMARY
# ============================================================

final_labels = [
    str(x["complexity"]).lower()
    for x in corrected
]

print("\n" + "=" * 80)
print("CORRECTED DATASET CREATED")
print("=" * 80)

print(f"\nTotal examples:        {len(corrected)}")
print(f"Reviewed examples:     {len(review)}")
print(f"Corrections applied:   {changes}")

print("\nFinal label distribution:")
print(Counter(final_labels))

print("\nOutput:")
print(OUTPUT_JSON)

print("\nOriginal V1 labels remain untouched:")
print(ORIGINAL_JSON)

CHECKING HUMAN REVIEW
Original examples: 639
Review rows:       125

All review labels are valid.

CORRECTED DATASET CREATED

Total examples:        639
Reviewed examples:     125
Corrections applied:   94

Final label distribution:
Counter({'medium': 307, 'complex': 213, 'simple': 119})

Output:
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v1_corrected_complexity_labels.json

Original V1 labels remain untouched:
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v1_complexity_labels.json


In [4]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# PATHS
# ============================================================

BASE = Path("/home/sagemaker-user/adaptive-llm-routing")

DATA_PATH = (
    BASE / "notebooks/data/evaluation/"
    "v1_corrected_complexity_labels.json"
)

# ============================================================
# LOAD CORRECTED DATASET
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 80)
print("V1.1 — HUMAN-CORRECTED LABEL TRAINING")
print("=" * 80)

print(f"\nDataset size: {len(data)}")

# ============================================================
# EXTRACT TEXT + LABELS
# ============================================================

texts = []
labels = []

for item in data:
    instruction = item.get("instruction", "").strip()
    complexity = item.get("complexity", "").strip().lower()

    if not instruction:
        continue

    if complexity not in {"simple", "medium", "complex"}:
        raise ValueError(
            f"Invalid complexity label: {complexity}"
        )

    texts.append(instruction)
    labels.append(complexity)

print(f"Usable examples: {len(texts)}")

print("\nLabel distribution:")
print(Counter(labels))

# ============================================================
# SAME SPLIT AS V1
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("\nSplit:")
print(f"Training:   {len(X_train)}")
print(f"Validation: {len(X_test)}")

# ============================================================
# SAME V1 TF-IDF CONFIGURATION
# ============================================================

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF:")
print(f"Training shape:   {X_train_tfidf.shape}")
print(f"Validation shape: {X_test_tfidf.shape}")

# ============================================================
# SAME V1 LOGISTIC REGRESSION
# ============================================================

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)

# ============================================================
# EVALUATION
# ============================================================

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 80)
print("V1.1 RESULTS")
print("=" * 80)

print(f"\nV1.1 Accuracy: {accuracy:.4f}")
print(f"V1.1 Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=["simple", "medium", "complex"],
        digits=4
    )
)

print("\nConfusion Matrix:")
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["simple", "medium", "complex"]
)

print(
    pd.DataFrame(
        cm,
        index=["Actual Simple", "Actual Medium", "Actual Complex"],
        columns=["Pred Simple", "Pred Medium", "Pred Complex"]
    )
)

# ============================================================
# COMPARE AGAINST ORIGINAL V1
# ============================================================

V1_ACCURACY = 0.6016

difference = accuracy - V1_ACCURACY

print("\n" + "=" * 80)
print("V1 vs V1.1")
print("=" * 80)

print(f"\nV1:      {V1_ACCURACY * 100:.2f}%")
print(f"V1.1:    {accuracy * 100:.2f}%")
print(f"Change:  {difference * 100:+.2f} percentage points")

if difference > 0:
    relative = (difference / V1_ACCURACY) * 100
    print(f"Relative improvement: {relative:+.2f}%")
    print("\n✓ V1.1 IMPROVED over V1")
elif difference < 0:
    relative = (difference / V1_ACCURACY) * 100
    print(f"Relative change: {relative:+.2f}%")
    print("\n✗ V1.1 is lower than V1")
else:
    print("\n= V1.1 is unchanged from V1")

# ============================================================
# SAVE MODEL ARTIFACTS
# ============================================================

import joblib

MODEL_DIR = BASE / "models" / "v1_1"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    vectorizer,
    MODEL_DIR / "tfidf_vectorizer.joblib"
)

joblib.dump(
    model,
    MODEL_DIR / "logistic_regression.joblib"
)

print("\nModel artifacts saved to:")
print(MODEL_DIR)

V1.1 — HUMAN-CORRECTED LABEL TRAINING

Dataset size: 639
Usable examples: 639

Label distribution:
Counter({'medium': 307, 'complex': 213, 'simple': 119})

Split:
Training:   511
Validation: 128

TF-IDF:
Training shape:   (511, 532)
Validation shape: (128, 532)

V1.1 RESULTS

V1.1 Accuracy: 0.4141
V1.1 Accuracy: 41.41%

Classification Report:
              precision    recall  f1-score   support

      simple     0.2174    0.2083    0.2128        24
      medium     0.4648    0.5410    0.5000        61
     complex     0.4412    0.3488    0.3896        43

    accuracy                         0.4141       128
   macro avg     0.3745    0.3661    0.3675       128
weighted avg     0.4105    0.4141    0.4091       128


Confusion Matrix:
                Pred Simple  Pred Medium  Pred Complex
Actual Simple             5           17             2
Actual Medium            11           33            17
Actual Complex            7           21            15

V1 vs V1.1

V1:      60.16%
V1.1: 

In [5]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sentence_transformers import SentenceTransformer


# ============================================================
# PATHS
# ============================================================

BASE = Path("/home/sagemaker-user/adaptive-llm-routing")

DATA_PATH = (
    BASE / "notebooks/data/evaluation/"
    "v1_complexity_labels.json"
)


# ============================================================
# LOAD ORIGINAL V1 DATA
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

texts = []
labels = []

for item in data:

    text = item.get("instruction", "").strip()
    label = item.get("complexity", "").strip().lower()

    if text and label in {"simple", "medium", "complex"}:
        texts.append(text)
        labels.append(label)


print("=" * 80)
print("V2 — SEMANTIC + LEXICAL + QUERY FEATURES")
print("=" * 80)

print(f"\nExamples: {len(texts)}")
print("Labels:", Counter(labels))


# ============================================================
# SAME SPLIT AS V1
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print(f"\nTraining:   {len(X_train)}")
print(f"Validation: {len(X_test)}")


# ============================================================
# TF-IDF FEATURES
# ============================================================

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF:")
print(X_train_tfidf.shape)


# ============================================================
# SEMANTIC EMBEDDINGS
# ============================================================

print("\nLoading MiniLM...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Encoding training data...")

X_train_emb = embedding_model.encode(
    X_train,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Encoding validation data...")

X_test_emb = embedding_model.encode(
    X_test,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbedding shape:")
print(X_train_emb.shape)


# ============================================================
# QUERY-LEVEL FEATURES
# ============================================================

def extract_query_features(texts):

    features = []

    for text in texts:

        words = text.split()

        word_count = len(words)

        char_count = len(text)

        sentence_count = max(
            1,
            sum(
                text.count(x)
                for x in [".", "?", "!"]
            )
        )

        question_mark = int("?" in text)

        newline_count = text.count("\n")

        avg_word_length = (
            np.mean([len(w) for w in words])
            if words else 0
        )

        features.append([
            word_count,
            char_count,
            sentence_count,
            question_mark,
            newline_count,
            avg_word_length
        ])

    return np.asarray(features, dtype=np.float32)


X_train_stats = extract_query_features(X_train)
X_test_stats = extract_query_features(X_test)

scaler = StandardScaler()

X_train_stats = scaler.fit_transform(X_train_stats)
X_test_stats = scaler.transform(X_test_stats)

print("\nQuery feature shape:")
print(X_train_stats.shape)


# ============================================================
# COMBINE FEATURES
# ============================================================

from scipy.sparse import csr_matrix, hstack

X_train_combined = hstack([
    X_train_tfidf,
    csr_matrix(X_train_emb),
    csr_matrix(X_train_stats)
])

X_test_combined = hstack([
    X_test_tfidf,
    csr_matrix(X_test_emb),
    csr_matrix(X_test_stats)
])

print("\nCombined feature shape:")
print(X_train_combined.shape)


# ============================================================
# V2 CLASSIFIER
# ============================================================

model = LogisticRegression(
    max_iter=1500,
    class_weight="balanced",
    random_state=42
)

print("\nTraining V2...")

model.fit(
    X_train_combined,
    y_train
)


# ============================================================
# EVALUATION
# ============================================================

y_pred = model.predict(X_test_combined)

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "=" * 80)
print("V2 RESULTS")
print("=" * 80)

print(f"\nV2 Accuracy: {accuracy:.4f}")
print(f"V2 Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=["simple", "medium", "complex"],
        digits=4
    )
)

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["simple", "medium", "complex"]
)

print(
    pd.DataFrame(
        cm,
        index=[
            "Actual Simple",
            "Actual Medium",
            "Actual Complex"
        ],
        columns=[
            "Pred Simple",
            "Pred Medium",
            "Pred Complex"
        ]
    )
)


# ============================================================
# COMPARE WITH V1
# ============================================================

V1_ACCURACY = 0.6016

difference = accuracy - V1_ACCURACY

print("\n" + "=" * 80)
print("V1 vs V2")
print("=" * 80)

print(f"\nV1:  {V1_ACCURACY * 100:.2f}%")
print(f"V2:  {accuracy * 100:.2f}%")
print(f"\nChange: {difference * 100:+.2f} percentage points")

if difference > 0:
    print("\n✓ V2 IMPROVED over V1")
elif difference < 0:
    print("\n✗ V2 is below V1")
else:
    print("\n= V2 matches V1")

/home/sagemaker-user/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


V2 — SEMANTIC + LEXICAL + QUERY FEATURES

Examples: 639
Labels: Counter({'simple': 213, 'medium': 213, 'complex': 213})

Training:   511
Validation: 128

TF-IDF:
(511, 526)

Loading MiniLM...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1389.64it/s]


Encoding training data...


Batches: 100%|██████████| 16/16 [00:04<00:00,  3.60it/s]


Encoding validation data...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.40it/s]



Embedding shape:
(511, 384)

Query feature shape:
(511, 6)

Combined feature shape:
(511, 916)

Training V2...

V2 RESULTS

V2 Accuracy: 0.6172
V2 Accuracy: 61.72%

Classification Report:
              precision    recall  f1-score   support

      simple     0.6087    0.6512    0.6292        43
      medium     0.5800    0.6744    0.6237        43
     complex     0.6875    0.5238    0.5946        42

    accuracy                         0.6172       128
   macro avg     0.6254    0.6165    0.6158       128
weighted avg     0.6249    0.6172    0.6160       128


Confusion Matrix:
                Pred Simple  Pred Medium  Pred Complex
Actual Simple            28            9             6
Actual Medium            10           29             4
Actual Complex            8           12            22

V1 vs V2

V1:  60.16%
V2:  61.72%

Change: +1.56 percentage points

✓ V2 IMPROVED over V1


In [6]:
# ============================================================
# V2 — INTERACTIVE ROUTING TEST
# ============================================================

def route_v2(query):
    # TF-IDF
    tfidf_features = tfidf.transform([query])

    # MiniLM embedding
    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Query statistics
    stats = extract_query_features([query])
    stats = scaler.transform(stats)

    # Combine
    features = hstack([
        tfidf_features,
        csr_matrix(embedding),
        csr_matrix(stats)
    ])

    # Prediction
    prediction = model.predict(features)[0]

    # Probabilities
    probabilities = model.predict_proba(features)[0]

    confidence = float(np.max(probabilities))

    probability_dict = {
        label: float(prob)
        for label, prob in zip(
            model.classes_,
            probabilities
        )
    }

    # Model routing
    routing_table = {
        "simple": "small_model",
        "medium": "medium_model",
        "complex": "large_model"
    }

    selected_model = routing_table[prediction]

    return {
        "query": query,
        "complexity": prediction,
        "confidence": confidence,
        "probabilities": probability_dict,
        "selected_model": selected_model
    }


# ============================================================
# TEST QUERIES
# ============================================================

test_queries = [
    "What is Python?",
    
    "What is the capital of France?",
    
    "Compare Python and Java for backend development.",
    
    "Explain the advantages and disadvantages of remote work.",
    
    "Analyze the tradeoffs between using a relational database and a NoSQL database for a large-scale application.",
    
    "Design a machine learning system that detects financial fraud while minimizing false positives and explain the tradeoffs.",
    
    "What is 25 times 4?",
    
    "Summarize the main differences between supervised and unsupervised learning.",
]


# ============================================================
# RUN TESTS
# ============================================================

for query in test_queries:

    result = route_v2(query)

    print("\n" + "=" * 80)
    print("QUERY:")
    print(result["query"])

    print(f"\nComplexity:     {result['complexity']}")
    print(f"Confidence:     {result['confidence']:.4f}")
    print(f"Selected model: {result['selected_model']}")

    print("\nProbabilities:")

    for label, probability in result["probabilities"].items():
        print(f"  {label:8s}: {probability:.4f}")


QUERY:
What is Python?

Complexity:     simple
Confidence:     0.5078
Selected model: small_model

Probabilities:
  complex : 0.2266
  medium  : 0.2655
  simple  : 0.5078

QUERY:
What is the capital of France?

Complexity:     complex
Confidence:     0.6696
Selected model: large_model

Probabilities:
  complex : 0.6696
  medium  : 0.2037
  simple  : 0.1267

QUERY:
Compare Python and Java for backend development.

Complexity:     simple
Confidence:     0.4874
Selected model: small_model

Probabilities:
  complex : 0.3816
  medium  : 0.1310
  simple  : 0.4874

QUERY:
Explain the advantages and disadvantages of remote work.

Complexity:     medium
Confidence:     0.4010
Selected model: medium_model

Probabilities:
  complex : 0.2377
  medium  : 0.4010
  simple  : 0.3613

QUERY:
Analyze the tradeoffs between using a relational database and a NoSQL database for a large-scale application.

Complexity:     complex
Confidence:     0.5331
Selected model: large_model

Probabilities:
  complex :

In [7]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split

BASE = Path("/home/sagemaker-user/adaptive-llm-routing")

DATA_PATH = (
    BASE / "notebooks/data/evaluation/"
    "v1_complexity_labels.json"
)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

texts = []
labels = []

for item in data:
    text = item.get("instruction", "").strip()
    label = item.get("complexity", "").strip().lower()

    if text and label in {"simple", "medium", "complex"}:
        texts.append(text)
        labels.append(label)

# Same initial split used by V1/V2
X_dev, X_fresh, y_dev, y_fresh = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=123,
    stratify=labels
)

print("=" * 80)
print("FRESH V2 EVALUATION SET")
print("=" * 80)

print(f"\nDevelopment examples: {len(X_dev)}")
print(f"Fresh test examples:   {len(X_fresh)}")

print("\nFresh test distribution:")
print(Counter(y_fresh))

FRESH V2 EVALUATION SET

Development examples: 511
Fresh test examples:   128

Fresh test distribution:
Counter({'simple': 43, 'medium': 43, 'complex': 42})


In [8]:
# ============================================================
# V2 — FRESH TEST SET EVALUATION
# Single-cell benchmark
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from scipy.sparse import csr_matrix, hstack


# ============================================================
# PATHS
# ============================================================

BASE = Path("/home/sagemaker-user/adaptive-llm-routing")

DATA_PATH = (
    BASE / "notebooks/data/evaluation/"
    "v1_complexity_labels.json"
)

RESULTS_PATH = (
    BASE / "notebooks/data/evaluation/"
    "v2_fresh_test_results.json"
)


# ============================================================
# LOAD ORIGINAL DATA
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

texts = []
labels = []

for item in data:

    text = item.get("instruction", "").strip()
    label = item.get("complexity", "").strip().lower()

    if text and label in {"simple", "medium", "complex"}:
        texts.append(text)
        labels.append(label)


# ============================================================
# CREATE COMPLETELY SEPARATE FRESH TEST SET
# ============================================================

X_dev, X_fresh, y_dev, y_fresh = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=123,
    stratify=labels
)

print("=" * 80)
print("V2 — FRESH TEST SET EVALUATION")
print("=" * 80)

print(f"\nDevelopment examples: {len(X_dev)}")
print(f"Fresh test examples:   {len(X_fresh)}")

print("\nFresh test distribution:")
print(Counter(y_fresh))


# ============================================================
# TRANSFORM FRESH DATA USING EXISTING V2 COMPONENTS
# ============================================================

print("\n" + "-" * 80)
print("Creating TF-IDF features...")
print("-" * 80)

X_fresh_tfidf = tfidf.transform(X_fresh)

print(f"TF-IDF shape: {X_fresh_tfidf.shape}")


print("\n" + "-" * 80)
print("Creating MiniLM embeddings...")
print("-" * 80)

X_fresh_emb = embedding_model.encode(
    X_fresh,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embedding shape: {X_fresh_emb.shape}")


print("\n" + "-" * 80)
print("Creating query features...")
print("-" * 80)

X_fresh_stats = extract_query_features(X_fresh)

X_fresh_stats = scaler.transform(
    X_fresh_stats
)

print(f"Query feature shape: {X_fresh_stats.shape}")


# ============================================================
# COMBINE V2 FEATURES
# ============================================================

X_fresh_combined = hstack([
    X_fresh_tfidf,
    csr_matrix(X_fresh_emb),
    csr_matrix(X_fresh_stats)
])

print("\nCombined feature shape:")
print(X_fresh_combined.shape)


# ============================================================
# PREDICTION
# ============================================================

print("\n" + "-" * 80)
print("Running V2 predictions...")
print("-" * 80)

y_fresh_pred = model.predict(
    X_fresh_combined
)


# ============================================================
# ACCURACY
# ============================================================

fresh_accuracy = accuracy_score(
    y_fresh,
    y_fresh_pred
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 80)
print("V2 FRESH TEST RESULTS")
print("=" * 80)

print(f"\nFresh Test Accuracy: {fresh_accuracy:.4f}")
print(f"Fresh Test Accuracy: {fresh_accuracy * 100:.2f}%")


print("\nClassification Report:")

report = classification_report(
    y_fresh,
    y_fresh_pred,
    labels=["simple", "medium", "complex"],
    digits=4
)

print(report)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_fresh,
    y_fresh_pred,
    labels=["simple", "medium", "complex"]
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual Simple",
        "Actual Medium",
        "Actual Complex"
    ],
    columns=[
        "Pred Simple",
        "Pred Medium",
        "Pred Complex"
    ]
)

print("\nConfusion Matrix:")
print(cm_df)


# ============================================================
# COMPARE WITH V1 AND ORIGINAL V2 VALIDATION
# ============================================================

V1_ACCURACY = 0.6016
V2_VALIDATION_ACCURACY = 0.6172

print("\n" + "=" * 80)
print("V1 vs V2")
print("=" * 80)

print(
    f"\nV1 validation accuracy: "
    f"{V1_ACCURACY * 100:.2f}%"
)

print(
    f"V2 validation accuracy: "
    f"{V2_VALIDATION_ACCURACY * 100:.2f}%"
)

print(
    f"V2 fresh test accuracy: "
    f"{fresh_accuracy * 100:.2f}%"
)

print(
    f"\nV2 validation improvement: "
    f"{(V2_VALIDATION_ACCURACY - V1_ACCURACY) * 100:+.2f} pp"
)

print(
    f"V2 fresh-test improvement vs V1: "
    f"{(fresh_accuracy - V1_ACCURACY) * 100:+.2f} pp"
)


# ============================================================
# VERDICT
# ============================================================

print("\n" + "=" * 80)
print("VERDICT")
print("=" * 80)

if fresh_accuracy > V1_ACCURACY:

    print(
        "\n✓ V2 BEATS V1 ON THE FRESH TEST SET."
    )

    print(
        "V2 can be considered the improved routing baseline."
    )

elif fresh_accuracy == V1_ACCURACY:

    print(
        "\n= V2 MATCHES V1 ON THE FRESH TEST SET."
    )

    print(
        "The validation improvement may not generalize."
    )

else:

    print(
        "\n✗ V2 DOES NOT BEAT V1 ON THE FRESH TEST SET."
    )

    print(
        "The +1.56 pp validation improvement did not generalize."
    )


# ============================================================
# SAVE RESULTS
# ============================================================

results = {
    "v1_validation_accuracy": float(V1_ACCURACY),
    "v2_validation_accuracy": float(V2_VALIDATION_ACCURACY),
    "v2_fresh_test_accuracy": float(fresh_accuracy),
    "v2_validation_improvement_pp": float(
        (V2_VALIDATION_ACCURACY - V1_ACCURACY) * 100
    ),
    "v2_fresh_test_improvement_pp": float(
        (fresh_accuracy - V1_ACCURACY) * 100
    ),
    "fresh_test_size": len(X_fresh),
    "fresh_test_distribution": dict(
        Counter(y_fresh)
    ),
    "confusion_matrix": cm.tolist()
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        indent=2
    )

print("\nResults saved to:")
print(RESULTS_PATH)

V2 — FRESH TEST SET EVALUATION

Development examples: 511
Fresh test examples:   128

Fresh test distribution:
Counter({'simple': 43, 'medium': 43, 'complex': 42})

--------------------------------------------------------------------------------
Creating TF-IDF features...
--------------------------------------------------------------------------------
TF-IDF shape: (128, 526)

--------------------------------------------------------------------------------
Creating MiniLM embeddings...
--------------------------------------------------------------------------------


Batches: 100%|██████████| 4/4 [00:01<00:00,  2.85it/s]

Embedding shape: (128, 384)

--------------------------------------------------------------------------------
Creating query features...
--------------------------------------------------------------------------------
Query feature shape: (128, 6)

Combined feature shape:
(128, 916)

--------------------------------------------------------------------------------
Running V2 predictions...
--------------------------------------------------------------------------------

V2 FRESH TEST RESULTS

Fresh Test Accuracy: 0.8359
Fresh Test Accuracy: 83.59%

Classification Report:
              precision    recall  f1-score   support

      simple     0.7708    0.8605    0.8132        43
      medium     0.8250    0.7674    0.7952        43
     complex     0.9250    0.8810    0.9024        42

    accuracy                         0.8359       128
   macro avg     0.8403    0.8363    0.8369       128
weighted avg     0.8396    0.8359    0.8364       128


Confusion Matrix:
                Pred Si

In [9]:
# ============================================================
# V2 — INTERACTIVE ROUTER TEST
# ============================================================

def test_v2(query):

    # --------------------------------------------------------
    # 1. TF-IDF
    # --------------------------------------------------------
    tfidf_features = tfidf.transform([query])

    # --------------------------------------------------------
    # 2. MiniLM semantic embedding
    # --------------------------------------------------------
    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # --------------------------------------------------------
    # 3. Query-level features
    # --------------------------------------------------------
    stats = extract_query_features([query])
    stats = scaler.transform(stats)

    # --------------------------------------------------------
    # 4. Combine V2 features
    # --------------------------------------------------------
    features = hstack([
        tfidf_features,
        csr_matrix(embedding),
        csr_matrix(stats)
    ])

    # --------------------------------------------------------
    # 5. Prediction
    # --------------------------------------------------------
    prediction = model.predict(features)[0]

    probabilities = model.predict_proba(features)[0]

    probability_dict = {
        label: float(prob)
        for label, prob in zip(
            model.classes_,
            probabilities
        )
    }

    confidence = float(np.max(probabilities))

    # --------------------------------------------------------
    # 6. Routing decision
    # --------------------------------------------------------
    routing_table = {
        "simple": "small_model",
        "medium": "medium_model",
        "complex": "large_model"
    }

    selected_model = routing_table[prediction]

    # --------------------------------------------------------
    # 7. Display
    # --------------------------------------------------------
    print("\n" + "=" * 80)
    print("V2 ROUTER TEST")
    print("=" * 80)

    print(f"\nQuery:")
    print(query)

    print("\n" + "-" * 80)

    print(f"Predicted complexity : {prediction}")
    print(f"Confidence           : {confidence:.4f}")
    print(f"Selected model       : {selected_model}")

    print("\nClass probabilities:")

    for label in ["simple", "medium", "complex"]:
        print(
            f"  {label:<8}: "
            f"{probability_dict.get(label, 0):.4f}"
        )

    print("=" * 80)

    return {
        "query": query,
        "complexity": prediction,
        "confidence": confidence,
        "probabilities": probability_dict,
        "selected_model": selected_model
    }


# ============================================================
# TEST THESE EXAMPLES
# ============================================================

test_queries = [

    # SIMPLE
    "What is Python?",
    "What is the capital of France?",
    "What is 25 multiplied by 4?",
    "Extract the names of the three people from this sentence.",

    # MEDIUM
    "Compare Python and Java for backend development.",
    "Explain the advantages and disadvantages of working remotely.",
    "What are the main differences between supervised and unsupervised learning?",

    # COMPLEX
    "Design a fraud detection system for a bank that minimizes false positives while detecting previously unseen fraud patterns.",
    "Analyze the tradeoffs between cost, latency, accuracy, and scalability when choosing between several large language models for a production application.",
    "Design an AI system that processes millions of financial transactions in real time, detects fraud, explains its decisions, and adapts to changing fraud patterns."
]


# ============================================================
# RUN TESTS
# ============================================================

results = []

for query in test_queries:

    result = test_v2(query)
    results.append(result)


# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n\n" + "=" * 80)
print("V2 TEST SUMMARY")
print("=" * 80)

summary = pd.DataFrame([
    {
        "query": r["query"],
        "complexity": r["complexity"],
        "confidence": round(r["confidence"], 4),
        "selected_model": r["selected_model"]
    }
    for r in results
])

display(summary)


V2 ROUTER TEST

Query:
What is Python?

--------------------------------------------------------------------------------
Predicted complexity : simple
Confidence           : 0.5078
Selected model       : small_model

Class probabilities:
  simple  : 0.5078
  medium  : 0.2655
  complex : 0.2266

V2 ROUTER TEST

Query:
What is the capital of France?

--------------------------------------------------------------------------------
Predicted complexity : complex
Confidence           : 0.6696
Selected model       : large_model

Class probabilities:
  simple  : 0.1267
  medium  : 0.2037
  complex : 0.6696

V2 ROUTER TEST

Query:
What is 25 multiplied by 4?

--------------------------------------------------------------------------------
Predicted complexity : complex
Confidence           : 0.4884
Selected model       : large_model

Class probabilities:
  simple  : 0.3402
  medium  : 0.1714
  complex : 0.4884

V2 ROUTER TEST

Query:
Extract the names of the three people from this sentence.



,query,complexity,confidence,selected_model
0,What is Python?,simple,0.5078,small_model
1,What is the capital of France?,complex,0.6696,large_model
2,What is 25 multiplied by 4?,complex,0.4884,large_model
3,Extract the names of the three people from thi...,simple,0.5790,small_model
4,Compare Python and Java for backend development.,simple,0.4874,small_model
5,Explain the advantages and disadvantages of wo...,simple,0.4723,small_model
6,What are the main differences between supervis...,complex,0.4044,large_model
7,Design a fraud detection system for a bank tha...,complex,0.5254,large_model
8,"Analyze the tradeoffs between cost, latency, a...",complex,0.5023,large_model
9,Design an AI system that processes millions of...,medium,0.4395,medium_model


In [7]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU available to this kernel.")

PyTorch version: 2.14.1+cpu
CUDA available: False
GPU count: 0
No GPU available to this kernel.
